In [59]:
import sys
import os

# Get project root (one level above notebook/)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:", project_root)

Project root added: /Users/suriyaa/Documents/Projects/LLMOps/Flipkart Product Recommender


In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from flipkart.config import Config
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
model = ChatGroq(model='openai/gpt-oss-120b')
embedding_model = Config.EMBEDDING_MODEL
embeddings = HuggingFaceEndpointEmbeddings(model=embedding_model)

In [61]:
import os
from langchain_astradb import AstraDBVectorStore
from dotenv import load_dotenv

load_dotenv(override=True)

vstore = AstraDBVectorStore(
    embedding=embeddings,
    collection_name="flipkart_database",
    api_endpoint=os.getenv('ASTRA_DB_API_ENDPOINT'),
    token=os.getenv('ASTRA_DB_APPLICATION_TOKEN'),
    namespace=Config.ASTRA_DB_KEYSPACE
)



In [62]:
retriever = vstore.as_retriever()

# RAG without History

In [83]:
prompt = ChatPromptTemplate.from_messages([

    ('system', """

        You are an expert assistant. Use only the provided context. to answer the user query
        Guidelines:
        1. Use ONLY the information from the context above.
        2. If multiple pieces of information are relevant, combine them logically.
        3. If the answer is not explicitly stated, say:
        "The provided context does not contain sufficient information."
        4. Do NOT use prior knowledge.
        5. Keep the answer structured and precise.

        Context: {context}
    """),

    ('human',"""
    
        Question: {question}
        Answer: 
    """)



])

In [84]:
chain = ({
    "context": retriever,
    "question": RunnablePassthrough()
}
| prompt
| model
| StrOutputParser()
)

In [85]:
chain.invoke('Best earphones')

'**Earphones highlighted as “best” in the provided context**\n\n| Product (as listed) | Description indicating it is a top choice |\n|----------------------|--------------------------------------------|\n| OnePlus Bullets Wireless Z **Bass Edition** Bluetooth Headset | Described as “Best earphone.” |\n| BoAt BassHeads 100 Wired Headset | Called “Best budgeted earphones… Best earphones till used by me.” |\n| realme Buds 2 Wired Headset | Noted for “nice earphones… the build quality is best… better for bass lover.” |\n| OnePlus Bullets Wireless Z Bluetooth Headset | Referred to as “Really great earphones… I totally recommend this.” |\n\nThese four products are identified in the context as among the best earphones.'

'**Earphones highlighted as “best” in the provided context**\n\n| Product (as listed) | Description indicating it is a top choice |\n|----------------------|--------------------------------------------|\n| OnePlus Bullets Wireless Z **Bass Edition** Bluetooth Headset | Described as “Best earphone.” |\n| BoAt BassHeads 100 Wired Headset | Called “Best budgeted earphones… Best earphones till used by me.” |\n| realme Buds 2 Wired Headset | Noted for “nice earphones… the build quality is best… better for bass lover.” |\n| OnePlus Bullets Wireless Z Bluetooth Headset | Referred to as “Really great earphones… I totally recommend this.” |\n\nThese four products are identified in the context as among the best earphones.'

# RAG with History

## Testing Question Rewriter

In [86]:
context_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the chat history and user question, rewrite it as a standalone question."),
    MessagesPlaceholder(variable_name="chat_history"), 
    ("human", "{input}")
])

In [117]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_community.chat_message_histories import ChatMessageHistory

history_store = ChatMessageHistory()

history_store.messages.append(HumanMessage(content="What is Best earphone"))
history_store.messages.append(AIMessage(content="""'**Earphones highlighted as “best” in the provided context**\n\n| Product (as listed) | Description indicating it is a top choice |\n|----------------------|--------------------------------------------|\n| OnePlus Bullets Wireless Z **Bass Edition** Bluetooth Headset | Described as “Best earphone.” |\n| BoAt BassHeads 100 Wired Headset | Called “Best budgeted earphones… Best earphones till used by me.” |\n| realme Buds 2 Wired Headset | Noted for “nice earphones… the build quality is best… better for bass lover.” |\n| OnePlus Bullets Wireless Z Bluetooth Headset | Referred to as “Really great earphones… I totally recommend this.” |\n\nThese four products are identified in the context as among the best earphones.'"""))

In [123]:
history_aware_retriever = context_prompt | llm | StrOutputParser()

In [124]:
# Testing history aware retriever
response = history_aware_retriever.invoke({
    'input': 'Which has better base',
    'chat_history': history_store.messages
})
response

'Which of the earphones listed (OnePlus Bullets Wireless Z Bass Edition, BoAt BassHeads\u202f100, realme Buds\u202f2, and OnePlus Bullets Wireless Z) provides the better bass performance?'

## Testing Answer Generator

In [125]:
qa_prompt = ChatPromptTemplate.from_messages([
            ("system", """You're an e-commerce bot answering product-related queries using reviews and titles.
                          Stick to context. Be concise and helpful.\n\nCONTEXT:\n{context}\n\n"""),
            MessagesPlaceholder(variable_name="chat_history"), 
            ("human", "{input}")  
        ])

In [136]:
from operator import itemgetter

question_answer_chain = ({
    "context": itemgetter("input") | retriever,
    "input": itemgetter("input"),
    "chat_history": itemgetter("chat_history")
}
| qa_prompt
| llm
| StrOutputParser()
)

In [ ]:
# Invoke with BOTH keys
question_answer_chain.invoke({
    'input': response,
    'chat_history': history_store.messages
})

'**realme Buds\u202f2** provides the strongest bass among the four.\n\n- In the comparison list, realme Buds\u202f2 is ranked **#1 for “Excellent Extra Bass”**, ahead of the BoAt and Sennheiser models.  \n- The BoAt BassHeads\u202f100 are praised for good bass, but not singled out as the best.  \n- The OnePlus Bullets Wireless\u202fZ (both regular and “Bass Edition”) note an improvement in bass over earlier versions, yet they don’t claim the “extra‑bass” performance that realme Buds\u202f2 does.\n\nSo, if bass performance is your top priority, the realme Buds\u202f2 is the clear winner.'

'**realme Buds\u202f2** provides the strongest bass among the four.\n\n- In the comparison list, realme Buds\u202f2 is ranked **#1 for “Excellent Extra Bass”**, ahead of the BoAt and Sennheiser models.  \n- The BoAt BassHeads\u202f100 are praised for good bass, but not singled out as the best.  \n- The OnePlus Bullets Wireless\u202fZ (both regular and “Bass Edition”) note an improvement in bass over earlier versions, yet they don’t claim the “extra‑bass” performance that realme Buds\u202f2 does.\n\nSo, if bass performance is your top priority, the realme Buds\u202f2 is the clear winner.'

## Combining the Question Rewriter & Answer Generator

In [144]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Step 1: Rewrite the question using history
history_aware_retriever = context_prompt | llm | StrOutputParser()

# Step 2: Answer using rewritten question + retriever
question_answer_chain = (
    {
        "context": itemgetter("input") | retriever,
        "input": itemgetter("input"),
        "chat_history": itemgetter("chat_history")
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

# Step 3: Combine — the bridge lambda reshapes the rewritten question back into a dict
def rewrite_and_pass(input_dict):
    rewritten = history_aware_retriever.invoke(input_dict)
    return {
        "input": rewritten,                      # rephrased standalone question
        "chat_history": input_dict["chat_history"]  # preserve history for qa_prompt
    }

conversational_rag_chain = RunnableLambda(rewrite_and_pass) | question_answer_chain

In [146]:
# First turn
answer1 = conversational_rag_chain.invoke({
    "input": "Which has better bass?",
    "chat_history": history_store.messages
})

# Update history after each turn
history_store.add_user_message("Which has better bass?")
history_store.add_ai_message(answer1)

# Second turn — history is automatically carried forward
answer2 = conversational_rag_chain.invoke({
    "input": "What is the price range?",
    "chat_history": history_store.messages
})

history_store.add_user_message("What is the price range?")
history_store.add_ai_message(answer2)


In [147]:
history_store.messages

[HumanMessage(content='What is Best earphone', additional_kwargs={}, response_metadata={}),
 AIMessage(content="'**Earphones highlighted as “best” in the provided context**\n\n| Product (as listed) | Description indicating it is a top choice |\n|----------------------|--------------------------------------------|\n| OnePlus Bullets Wireless Z **Bass Edition** Bluetooth Headset | Described as “Best earphone.” |\n| BoAt BassHeads 100 Wired Headset | Called “Best budgeted earphones… Best earphones till used by me.” |\n| realme Buds 2 Wired Headset | Noted for “nice earphones… the build quality is best… better for bass lover.” |\n| OnePlus Bullets Wireless Z Bluetooth Headset | Referred to as “Really great earphones… I totally recommend this.” |\n\nThese four products are identified in the context as among the best earphones.'", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Which has better bass?', additional_kwargs={}, response_m